# Stop Delay Change (Polars)

Compares delay changes by next stop, or by city part when a stop-to-city-part mapping is available, using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

stop_delay_change = load_polars_script("polars_stop_delay_change", "stop-delay-change.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
TIMEZONE = "Europe/Helsinki"
GTFS_DIR = None
GTFS_ROOT = PROJECT_ROOT / "data" / "gtfs"
LIMIT = 20
MIN_OBSERVATIONS = 100
GROUP_BY = "stop"  # "stop" or "city-part"
CITY_PARTS_CSV = None  # Example: PROJECT_ROOT / "data" / "stop-city-parts.csv"
SORT_BY = "absolute"  # "absolute", "increase", or "decrease"
LINE_REF = None
DEFAULT_PERIOD_DAYS = 1
BASELINE_START, BASELINE_END, COMPARISON_START, COMPARISON_END = stop_delay_change.default_recent_periods(
    DB,
    timezone=TIMEZONE,
    period_days=DEFAULT_PERIOD_DAYS,
)
DIRECTION_REF = None
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"
LEGACY_MIDPOINT = False


By default this notebook uses two recent one-day periods derived from the latest observation timestamp. Override all four period variables for matched custom windows.

Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    timezone = TIMEZONE
    gtfs_dir = GTFS_DIR
    gtfs_root = GTFS_ROOT
    city_parts_csv = CITY_PARTS_CSV
    group_by = GROUP_BY
    sort_by = SORT_BY
    line_ref = LINE_REF
    direction_ref = DIRECTION_REF
    baseline_start = BASELINE_START
    baseline_end = BASELINE_END
    comparison_start = COMPARISON_START
    comparison_end = COMPARISON_END
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    legacy_midpoint = LEGACY_MIDPOINT

buckets = stop_delay_change.load_delay_buckets_for_args(Args)
change, period_description = stop_delay_change.build_stop_change_from_buckets(Args, buckets)
print(period_description)
change


baseline=2026-05-06T13:20:01+00:00..2026-05-07T13:20:01+00:00, comparison=2026-05-07T13:20:01+00:00..2026-05-08T13:20:01+00:00


stop_id,baseline_bucket_count,baseline_raw_poll_count,baseline_signed_mean_delay_min,baseline_avg_delay_min,baseline_median_delay_min,baseline_p75_delay_min,baseline_p90_delay_min,baseline_p95_delay_min,baseline_pct_late,baseline_pct_over_3_min_late,baseline_pct_over_5_min_late,baseline_pct_early,baseline_pct_over_1_min_early,baseline_pct_over_3_min_early,baseline_median_early_min_abs,baseline_p90_early_min_abs,stop_name,city_part,stop_lat,stop_lon,comparison_bucket_count,comparison_raw_poll_count,comparison_signed_mean_delay_min,comparison_avg_delay_min,comparison_median_delay_min,comparison_p75_delay_min,comparison_p90_delay_min,comparison_p95_delay_min,comparison_pct_late,comparison_pct_over_3_min_late,comparison_pct_over_5_min_late,comparison_pct_early,comparison_pct_over_1_min_early,comparison_pct_over_3_min_early,comparison_median_early_min_abs,comparison_p90_early_min_abs
str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,null,f64,f64,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [3]:
if change.is_empty():
    print("No matching observations found.")
else:
    label_column = "city_part" if GROUP_BY == "city-part" else "stop_name"
    if label_column not in change.columns:
        label_column = "city_part" if GROUP_BY == "city-part" else "stop_id"
    fig = px.bar(
        change.sort("p90_delay_change_min"),
        x="p90_delay_change_min",
        y=label_column,
        orientation="h",
        title="Delay change by stop or city part",
        labels={
            label_column: "City part" if GROUP_BY == "city-part" else "Stop",
            "p90_delay_change_min": "Comparison minus baseline P90 delay, minutes",
        },
    )
    fig.update_layout(showlegend=False)
    fig.show()


No matching observations found.
